# 1. Import and Hardware Setup

In [ ]:
import torch
import torchvision
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

import matplotlib.pyplot as plt
import time
import copy

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
DATA_PATH = './Data'
SAVE_PATH = './Model'

# 2. Hyperparameters

In [ ]:
BATCH_SIZE = 128
IMG_SIZE = 224
IMG_CHANNELS = 3

EPOCHS = 30
LR = 1e-3


# 3. Dataset Preparation

In [ ]:
stats = ((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.AutoAugment(transforms.AutoAugmentPolicy.IMAGENET),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(*stats)
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(*stats)
])

In [ ]:
# Load the full train data
full_train_data = datasets.Food101(root=DATA_PATH, split='train', transform=train_transform)

# Split the full train data into train and validation data
train_size = int(0.8 * len(full_train_data))
val_size = len(full_train_data) - train_size

train_subset, val_subset = random_split(full_train_data, (train_size, val_size))

# Change transform von val_subset to test_transform
val_subset.dataset = copy.copy(full_train_data)
val_subset.dataset.transform = test_transform

# Load Test Dataset
test_dataset = datasets.Food101(root=DATA_PATH, split='test', transform=test_transform)

In [ ]:
# Loader
train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True,
                        num_workers=4, pin_memory=True)

val_loader = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=True,
                        num_workers=4, pin_memory=True)

test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=True,
                        num_workers=4, pin_memory=True)



# 4. AlexNet Architecture

In [ ]:
class CNN(nn.Module):
    def _